# Modelagem com PyTorch — MLP para Predição de Churn

Notebook dedicado à construção, treinamento e avaliação de uma **Rede Neural (MLP)** utilizando PyTorch para o problema de churn.

**Etapas:**
1. Preparação dos Dados (replicando pipeline do `eda.ipynb`)
2. Arquitetura da MLP (PyTorch)
3. Loop de Treinamento com Batching e Early Stopping
4. Comparação MLP vs Modelos Baseline (6 métricas)
5. Análise de Trade-off de Custo: Falsos Positivos vs Falsos Negativos
6. Registro de Experimentos no MLflow

## 1. Setup e Preparação dos Dados

In [10]:
# Bibliotecas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import (
    SelectKBest, chi2, mutual_info_classif, f_classif,
    RFE, SelectFromModel
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_curve
)

# # XGBoost / LightGBM
# from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# MLflow
import mlflow
import mlflow.pytorch
import mlflow.sklearn

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")

Dispositivo: cpu


### 1.1 Carregamento e Pré-processamento

Replicamos o mesmo pipeline de dados do `eda.ipynb` para garantir comparabilidade.

In [15]:
# ── Carregar dados ────────────────────────────────────────────────
df_churn = pd.read_csv("../data/Telco_customer_churn.csv")
print(f"Shape original: {df_churn.shape}")

# ── Remover colunas iniciais ──────────────────────────────────────
df_churn = df_churn.drop(columns=['Count', 'CustomerID', 'Churn Reason'])

# ── Feature Engineering ───────────────────────────────────────────
# 1. Perfil de alto risco: Fibra óptica + Contrato mensal
df_churn['high_risk_profile'] = (
    (df_churn['Internet Service'] == 'Fiber optic') &
    (df_churn['Contract'] == 'Month-to-month')
).astype(int)

# 2. Idoso isolado: Senior + Sem parceiro + Sem dependentes
df_churn['isolated_senior'] = (
    (df_churn['Senior Citizen'] == 'Yes') &
    (df_churn['Partner'] == 'No') &
    (df_churn['Dependents'] == 'No')
).astype(int)

# 3. Contagem de serviços de internet contratados
servicos = ['Online Security', 'Online Backup', 'Device Protection',
            'Tech Support', 'Streaming TV', 'Streaming Movies']
df_churn['internet_services_count'] = sum(
    (df_churn[c] == 'Yes').astype(int) for c in servicos
)

# 4. Custo por mês de permanência
df_churn['cost_per_month'] = df_churn['Monthly Charges'] / (df_churn['Tenure Months'] + 1)

# ── Conversão de tipos ────────────────────────────────────────────
for col in df_churn.columns:
    try:
        df_churn[col] = pd.to_numeric(df_churn[col])
    except (ValueError, TypeError):
        pass

# Total Charges: converter para float e preencher nulos
df_churn['Total Charges'] = pd.to_numeric(df_churn['Total Charges'], errors='coerce')
df_churn['Total Charges'] = df_churn['Total Charges'].fillna(df_churn['Total Charges'].median())

# ── Remover colunas de leakage e geográficas ─────────────────────
colunas_vazar = ['Churn Score', 'CLTV', 'Churn Label']
colunas_geograficas = ['City', 'Country', 'Lat Long', 'Latitude', 'Longitude', 'Zip Code']
colunas_drop = colunas_vazar + colunas_geograficas
df_churn = df_churn.drop(columns=[c for c in colunas_drop if c in df_churn.columns])

# ── One-Hot Encoding ──────────────────────────────────────────────
cat_cols = df_churn.select_dtypes(include=['object', 'category']).columns.tolist()
data_encoded = pd.get_dummies(df_churn, columns=cat_cols, drop_first=False)
bool_cols = data_encoded.select_dtypes(include=bool).columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

# ── Preparação X e y ──────────────────────────────────────────────
df_model = data_encoded.select_dtypes(include=[np.number]).copy()
X = df_model.drop(columns=['Churn Value'])
y = df_model['Churn Value']
X = X.fillna(X.median())

print(f"Features disponíveis: {X.shape[1]}")
print(f"Distribuição do target:\n{y.value_counts(normalize=True).round(4)}")

Shape original: (7043, 33)
Features disponíveis: 51
Distribuição do target:
Churn Value
0    0.7346
1    0.2654
Name: proportion, dtype: float64


### 1.1 Carregamento e Pré-processamento

Replicamos o mesmo pipeline de dados do `eda.ipynb` para garantir comparabilidade.

In [21]:
# ── Carregar dados ────────────────────────────────────────────────
df_churn = pd.read_csv("../data/Telco_customer_churn.csv")
print(f"Shape original: {df_churn.shape}")

# ── Remover colunas iniciais ──────────────────────────────────────
df_churn = df_churn.drop(columns=['Count', 'CustomerID', 'Churn Reason'])

# ── Feature Engineering ───────────────────────────────────────────
# 1. Perfil de alto risco: Fibra óptica + Contrato mensal
df_churn['high_risk_profile'] = (
    (df_churn['Internet Service'] == 'Fiber optic') &
    (df_churn['Contract'] == 'Month-to-month')
).astype(int)

# 2. Idoso isolado: Senior + Sem parceiro + Sem dependentes
df_churn['isolated_senior'] = (
    (df_churn['Senior Citizen'] == 'Yes') &
    (df_churn['Partner'] == 'No') &
    (df_churn['Dependents'] == 'No')
).astype(int)

# 3. Contagem de serviços de internet contratados
servicos = ['Online Security', 'Online Backup', 'Device Protection',
            'Tech Support', 'Streaming TV', 'Streaming Movies']
df_churn['internet_services_count'] = sum(
    (df_churn[c] == 'Yes').astype(int) for c in servicos
)

# 4. Custo por mês de permanência
df_churn['cost_per_month'] = df_churn['Monthly Charges'] / (df_churn['Tenure Months'] + 1)

# ── Conversão de tipos ────────────────────────────────────────────
for col in df_churn.columns:
    try:
        df_churn[col] = pd.to_numeric(df_churn[col])
    except (ValueError, TypeError):
        pass

# Total Charges: converter para float e preencher nulos
df_churn['Total Charges'] = pd.to_numeric(df_churn['Total Charges'], errors='coerce')
df_churn['Total Charges'] = df_churn['Total Charges'].fillna(df_churn['Total Charges'].median())

# ── Remover colunas de leakage e geográficas ─────────────────────
colunas_vazar = ['Churn Score', 'CLTV', 'Churn Label']
colunas_geograficas = ['City', 'Country', 'Lat Long', 'Latitude', 'Longitude', 'Zip Code']
colunas_drop = colunas_vazar + colunas_geograficas
df_churn = df_churn.drop(columns=[c for c in colunas_drop if c in df_churn.columns])

# ── One-Hot Encoding ──────────────────────────────────────────────
cat_cols = df_churn.select_dtypes(include=['object', 'category']).columns.tolist()
data_encoded = pd.get_dummies(df_churn, columns=cat_cols, drop_first=False)
bool_cols = data_encoded.select_dtypes(include=bool).columns
data_encoded[bool_cols] = data_encoded[bool_cols].astype(int)

# ── Preparação X e y ──────────────────────────────────────────────
df_model = data_encoded.select_dtypes(include=[np.number]).copy()
X = df_model.drop(columns=['Churn Value'])
y = df_model['Churn Value']
X = X.fillna(X.median())

print(f"Features disponíveis: {X.shape[1]}")
print(f"Distribuição do target:\n{y.value_counts(normalize=True).round(4)}")

Shape original: (7043, 33)
Features disponíveis: 51
Distribuição do target:
Churn Value
0    0.7346
1    0.2654
Name: proportion, dtype: float64


In [20]:
# ── Train/Test Split ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

# ── Seleção de Features (Top 30 — mesmo processo do eda.ipynb) ──
K = 30
scaler_mm = MinMaxScaler()
X_scaled = pd.DataFrame(scaler_mm.fit_transform(X_train), columns=X_train.columns)

# 1. SelectKBest — ANOVA
skb_f = SelectKBest(score_func=f_classif, k=K)
skb_f.fit(X_scaled, y_train)
anova_scores = pd.Series(skb_f.scores_, index=X_train.columns)

# 2. SelectKBest — Chi2
skb_chi2 = SelectKBest(score_func=chi2, k=K)
skb_chi2.fit(X_scaled, y_train)
chi2_scores = pd.Series(skb_chi2.scores_, index=X_train.columns)

# 3. Mutual Information
mi = mutual_info_classif(X_scaled, y_train, random_state=SEED)
mi_scores = pd.Series(mi, index=X_train.columns)

# 4. Correlação Pearson
corr_target = X_train.corrwith(y_train).abs()

# 5. Random Forest Feature Importance
rf_temp = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_temp.fit(X_train, y_train)
rf_scores = pd.Series(rf_temp.feature_importances_, index=X_train.columns)

# 6. RFE com Logistic Regression
lr_temp = LogisticRegression(max_iter=2000, random_state=SEED)
rfe = RFE(estimator=lr_temp, n_features_to_select=K)
rfe.fit(X_scaled, y_train)
rfe_ranking = pd.Series(rfe.ranking_, index=X_train.columns)

# 7. SelectFromModel
selector_rf = SelectFromModel(rf_temp, threshold='mean', prefit=True)
sfm_mask = pd.Series(selector_rf.get_support().astype(int), index=X_train.columns)

# ── Ranking Agregado ──────────────────────────────────────────────
all_methods = pd.DataFrame({
    'anova':    anova_scores,
    'chi2':     chi2_scores,
    'mi':       mi_scores,
    'pearson':  corr_target,
    'rf_imp':   rf_scores,
    'rfe':      (rfe_ranking.max() - rfe_ranking + 1),  # inverter ranking
    'sfm':      sfm_mask,
})

# Normalizar cada método para [0, 1] e somar
all_norm = (all_methods - all_methods.min()) / (all_methods.max() - all_methods.min() + 1e-10)
ranking_final = all_norm.sum(axis=1).sort_values(ascending=False)

features_selecionadas = ranking_final.head(K).index.tolist()
print(f"\nTop {K} features selecionadas:")
print(features_selecionadas)

X_train_sel = X_train[features_selecionadas]
X_test_sel = X_test[features_selecionadas]

print(f"\nShape treino: {X_train_sel.shape} | Shape teste: {X_test_sel.shape}")

X_train: (5634, 51) | X_test: (1409, 51)

Top 30 features selecionadas:
['cost_per_month', 'high_risk_profile', 'Contract_Month-to-month', 'Tenure Months', 'Online Security_No', 'Tech Support_No', 'Payment Method_Electronic check', 'Monthly Charges', 'Total Charges', 'Dependents_No', 'Contract_Two year', 'Internet Service_Fiber optic', 'Dependents_Yes', 'Online Security_No internet service', 'Device Protection_No internet service', 'Streaming Movies_No internet service', 'Online Backup_No internet service', 'Streaming TV_No internet service', 'Tech Support_No internet service', 'Internet Service_No', 'Paperless Billing_No', 'Contract_One year', 'internet_services_count', 'Online Security_Yes', 'Online Backup_No', 'Tech Support_Yes', 'Partner_No', 'isolated_senior', 'Device Protection_No', 'Internet Service_DSL']

Shape treino: (5634, 30) | Shape teste: (1409, 30)


In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)

print(f"Média das features (treino): {X_train_scaled.mean(axis=0)[:5].round(6)}")
print(f"Desvio padrão (treino): {X_train_scaled.std(axis=0)[:5].round(6)}")

Média das features (treino): [ 0.  0.  0. -0.  0.]
Desvio padrão (treino): [1. 1. 1. 1. 1.]


---
## 2. Arquitetura da MLP (PyTorch)

Rede Neural do tipo **Multi-Layer Perceptron** com:
- 3 camadas ocultas (64 → 32 → 16 neurônios)
- Ativação **ReLU** nas camadas ocultas
- **Batch Normalization** para estabilizar o treinamento
- **Dropout** (0.3) para regularização
- Ativação **Sigmoid** na saída (classificação binária)
- Função de perda: **BCELoss** (Binary Cross-Entropy)
- Otimizador: **Adam**

In [23]:
class ChurnDataset(Dataset):
    """Dataset PyTorch para dados tabulares de churn."""

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y.values if hasattr(y, 'values') else y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [24]:
class ChurnMLP(nn.Module):
    """
    Multi-Layer Perceptron para classificação binária de churn.

    Arquitetura:
        Input (n_features)
        → Linear(64) → BatchNorm → ReLU → Dropout(0.3)
        → Linear(32) → BatchNorm → ReLU → Dropout(0.3)
        → Linear(16) → BatchNorm → ReLU → Dropout(0.3)
        → Linear(1)  → Sigmoid
    """

    def __init__(self, n_features, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            # Camada 1: input → 64
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Camada 2: 64 → 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Camada 3: 32 → 16
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Saída: 16 → 1
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


# Instanciar modelo
n_features = X_train_scaled.shape[1]
modelo_mlp = ChurnMLP(n_features).to(DEVICE)
print(modelo_mlp)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in modelo_mlp.parameters()):,}")

ChurnMLP(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=16, bias=True)
    (9): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=16, out_features=1, bias=True)
    (13): Sigmoid()
  )
)

Total de parâmetros: 4,833


---
## 3. Loop de Treinamento com Batching e Early Stopping